In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType

catalog_name = 'iranian_transactions'

In [0]:
#In the status column for transactions_1, we need to change the value of '0' to 'failed'
df_bronze = spark.table(f'{catalog_name}.bronze.brz_transactions_1')
display(df_bronze)



In [0]:
#show distinct values in status column
df_bronze.select('status').distinct().show()

#Capitalize the values in status column 
df_silver=df_bronze.withColumn("status", F.initcap(F.col("status")))\
    .withColumn("status", F.regexp_replace(F.col("status"), "Failed", "Fail"))\
    .withColumn("status", F.regexp_replace(F.col("status"), "Succeed", "Success"))

df_silver.select('city').distinct().show()

In [0]:
#next, we need to convert time to a timestamp type, and then split the time from the date
df_silver = df_silver.withColumn("time", F.to_timestamp(F.col("time"), "yyyy-MM-dd HH:mm:ss"))\
    .withColumn("date", F.date_format(F.col("time"), "yyyy-MM-dd"))\
    .withColumn("hour", F.date_format(F.col("time"), "HH"))\
    .withColumn("minute", F.date_format(F.col("time"), "mm"))\
    .withColumn("second", F.date_format(F.col("time"), "ss"))

    # df_silver = df_silver.withColumn("time", F.to_timestamp(F.col("time"), "yyyy-MM-dd HH:mm:ss"))\
    # .withColumn("date_t", F.date_format(F.col("time"), "yyyy-MM-dd"))\
    # .withColumn("hour_t", F.date_format(F.col("time"), "HH"))\
    # .withColumn("minute_t", F.date_format(F.col("time"), "mm"))\
    # .withColumn("second_t", F.date_format(F.col("time"), "ss"))

display(df_silver)

In [0]:

df_silver.select('card_type').distinct().show()
#change card_type to capitalize, and rename some of the values
#df_silver = df_silver.withColumn("card_type", F.initcap(F.col("card_type")))\
 #   .withColumn("card_type", F.regexp_replace(F.col("card_type"), "Mastercard", "Master Card"))\
 #   .withColumn("card_type", F.regexp_replace(F.col("card_type"), "Mastcard", "Master Card"))\
 #   .withColumn("card_type", F.regexp_replace(F.col("card_type"), "Master-card", "Master Card"))\
#    .withColumn("card_type", F.regexp_replace(F.col("card_type"), "Vsa", "Visa Card"))\
#    .withColumn("card_type", F.regexp_replace(F.col("card_type"), "Visa", "Visa Card"))\
 #   .withColumn("card_type", F.regexp_replace(F.col("card_type"), "Visa Card Card", "Visa Card"))


card_type_map = {
    "Mastercard": "Master Card", "Mastcard": "Master Card", "Master-card": "Master Card",
    "Vsa": "Visa Card", "Visa": "Visa Card"
}
mapping_expr = F.create_map([F.lit(x) for pair in card_type_map.items() for x in pair])

df_silver = df_silver.withColumn("card_type", F.initcap(F.col("card_type")))
df_silver = df_silver.withColumn(
    "card_type",
    F.coalesce(mapping_expr[F.col("card_type")], F.col("card_type"))
)


# display(df_silver.select("amount").max())

#df_silver.select('card_type').distinct().show()

#df_silver.select('city').distinct().show()
#display(df_silver)

In [0]:
#check for the max value in the amount column
df_silver.select("amount").distinct().show()
df_silver.printSchema()

In [0]:
#split the time column into 2 columns, the date and time respectively
df_silver = df_silver.withColumn("date", F.date_format(F.col("time"), "yyyy-MM-dd"))\
    .withColumn("time", F.date_format(F.col("time"), "HH:mm:ss"))

display(df_silver)

In [0]:
#Get the distinct items in the city column
df_silver.select('city').distinct().show()
#First find and remove any leading and trailing spaces in the city column
df_silver = df_silver.withColumn("city", F.trim(F.col("city")))
#Next, change the values in the city column to capitalize
df_silver = df_silver.withColumn("city", F.initcap(F.col("city")))

#Replace the misspelled city names with the correct names
#df_silver = df_silver.withColumn("city", F.regexp_replace(F.col("city"), "Thr", "Tehran"))\
 #   .withColumn("city", F.regexp_replace(F.col("city"), "Tehr@n", "Tehran"))\
  #  .withColumn("city", F.regexp_replace(F.col("city"), "Thran", "Tehran"))\
   # .withColumn("city", F.regexp_replace(F.col("city"   ), "Tehranan", "Tehran"))

city_map = {
    "Thr": "Tehran", "Tehr@n": "Tehran", "Thran": "Tehran", "Tehranan": "Tehran","TEHRAN": "Tehran"
}
mapping_expr = F.create_map([F.lit(x) for pair in city_map.items() for x in pair])

df_silver = df_silver.withColumn("city", F.initcap(F.col("city")))
df_silver = df_silver.withColumn(
    "city",
    F.coalesce(mapping_expr[F.col("city")], F.col("city"))
)
df_silver.select('city').distinct().show()




In [0]:
display(df_silver)

known_cities = {"Tehran", "Isfahan", "Tabriz, Shiraz, Isfahan, Karaj, Ahvaz, Qom, Mashhad, Nan"}
unmapped = df_silver.filter(~F.col("city").isin(known_cities)).select("city").distinct()
if unmapped.count() > 0:
    unmapped.show()  # or log/raise — decide if these should fail the pipeline

In [0]:
#Display the negative values in the dataset, this would be tagged as debt later on
#df_silver.filter(df_silver.amount < 0).show()
#change the amount column to float type
#df_silver = df_silver.withColumn("amount", F.col("amount").cast(FloatType()))
df_silver = df_silver.withColumn("amount", F.col("amount").cast("float"))
#display(df_silver)

In [0]:
df_silver = df_silver.withColumn("is_debt", F.col("amount") < 0)

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_transactions")

Note: tr_1 =trx-10k csv, tr_2 = trx_99, tr_3=trx-extra-10k.csv, tr_4=trx.csv
Now, we have only been working on tr_1, so far. So le's do tr_2 now

In [0]:
df_bronze = spark.table(f'{catalog_name}.bronze.brz_transactions_2')
display(df_bronze)


In [0]:
#So check for nulls in the data
#df_bronze.filter(df_bronze.time.isNull()).display()
#df_bronze.filter(df_bronze.id.isNull()).display()


#let's first find the scema of each column in the dataset
df_bronze.printSchema()

df_bronze.select("city").distinct().show()



In [0]:
df_silver = df_bronze.withColumn("time_raw", F.col("time")) \
    .withColumn("time", F.try_to_timestamp(F.col("time"), F.lit("HH:mm yyyy-MM-dd")))


df_parsed = df_silver \
    .withColumn("time_parsed", F.try_to_timestamp(F.col("time"), F.lit("HH:mm yyyy-MM-dd"))) \
    .withColumn("amount_parsed", F.col("amount").cast(FloatType()))

MAX_PLAUSIBLE_AMOUNT = 5_000_000  # placeholder — see note below on picking this properly

df_flagged = df_silver.withColumn("amount_parsed", F.col("amount").cast(FloatType())) \
    .withColumn(
        "reject_reason",
        F.when(F.col("time_raw").isNotNull() & (F.col("time_raw") != "") & F.col("time").isNull(), "invalid_time")
         .when(F.col("amount").isNotNull() & F.col("amount_parsed").isNull(), "non_numeric_amount")
         .when(F.col("amount_parsed") > F.lit(MAX_PLAUSIBLE_AMOUNT), "amount_out_of_range")
         .otherwise(F.lit(None))
    )

df_rejected = df_flagged.filter(F.col("reject_reason").isNotNull())
df_clean    = df_flagged.filter(F.col("reject_reason").isNull()).drop("reject_reason")

df_rejected.write.format("delta").mode("append") \
    .saveAsTable(f"{catalog_name}.silver.slv_transactions_rejected")

df_clean.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_transactions_1")

In [0]:
#Let's change the time column to timestamp type
df_silver = df_silver.withColumn("time", F.to_timestamp(F.col("time"), "HH:mm yyyy-MM-dd"))\
    .withColumn("date", F.date_format(F.col("time"), "yyyy-MM-dd"))\
    .withColumn("hour", F.date_format(F.col("time"), "HH"))\
    .withColumn("minute", F.date_format(F.col("time"), "mm"))

# df_silver = df_silver.withColumn("time", F.to_timestamp(F.col("time"), "HH:mm yyyy-MM-dd"))\
#     .withColumn("date_t", F.date_format(F.col("time"), "yyyy-MM-dd"))\
#     .withColumn("hour_t", F.date_format(F.col("time"), "HH"))\
#     .withColumn("minute_t", F.date_format(F.col("time"), "mm"))
    
#Let's check the distinct values in the card_type column
df_silver.select('card_type').distinct().show()
#change card_type to capitalize, and rename some of the values
df_silver = df_silver.withColumn("card_type", F.initcap(F.col("card_type")))\
.withColumn("city", F.trim(F.col("city")))

card_type_map = {
    "Mastercard": "Master Card", "Mastcard": "Master Card", "Master-card": "Master Card",
    "Vsa": "Visa Card", "Visa": "Visa Card"
}
mapping_expr = F.create_map([F.lit(x) for pair in card_type_map.items() for x in pair])

df_silver = df_silver.withColumn("card_type", F.initcap(F.col("card_type")))
df_silver = df_silver.withColumn(
    "card_type",
    F.coalesce(mapping_expr[F.col("card_type")], F.col("card_type"))
)

#Let's check the distinct values in the city column
df_silver.select('city').distinct().show()

city_map = {
    "Thr": "Tehran", "Tehr@n": "Tehran", "Thran": "Tehran", "Tehranan": "Tehran","TEHRAN": "Tehran"
}
mapping_expr = F.create_map([F.lit(x) for pair in city_map.items() for x in pair])

df_silver = df_silver.withColumn("city", F.initcap(F.col("city")))
df_silver = df_silver.withColumn(
    "city",
    F.coalesce(mapping_expr[F.col("city")], F.col("city"))
)
df_silver.select('city').distinct().show()

#df_silver = df_silver.withColumn("amount", F.col("amount").cast("float"))
df_silver = df_silver.withColumn("amount", F.col("amount").cast("string"))
#change the id column to integer type
df_silver = df_silver.withColumn("id", F.col("id").cast(IntegerType()))
#Let's check the distinct values in the status column
df_silver.select('status').distinct().show()

status_map = {
    "failed": "Fail", "succeed": "Success", "FAIL": "Fail","Succeed": "Success","Failed": "Fail" 
}
mapping_expr = F.create_map([F.lit(x) for pair in status_map.items() for x in pair])

df_silver = df_silver.withColumn("status", F.initcap(F.col("status")))
df_silver = df_silver.withColumn(
    "status",
    F.coalesce(mapping_expr[F.col("status")], F.col("status"))
)
df_silver.select('status').distinct().show()

In [0]:
df_silver.select("time").printSchema()

root
 |-- time: timestamp (nullable = true)



In [0]:
#Now it's to create a column for debt where amount is less than 0
df_silver = df_silver.withColumn("amount", F.col("amount").cast("float"))\
    .withColumn("Debt", F.when(F.col("amount") < 0, F.col("amount")).otherwise(F.lit(None)))\
    .withColumn("Debt", F.col("Debt").cast("double"))

df_silver.printSchema()

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_transactions_2")

In [0]:
df_bronze = spark.table(f'{catalog_name}.bronze.brz_transactions_3')


#Capitalize the values in status column 
df_silver=df_bronze.withColumn("status", F.initcap(F.col("status")))\
    .withColumn("status", F.regexp_replace(F.col("status"), "Failed", "Fail"))\
    .withColumn("status", F.regexp_replace(F.col("status"), "Succeed", "Success"))

#df_silver.printSchema()
display(df_silver)

df_silver.select("city").distinct().show()


In [0]:

#df_silver = df_silver.withColumn("time", F.try_to_timestamp(F.col("time"), F.lit("yyyy-MM-dd HH:mm:ss ")))\
#                    .withColumn("date", F.date_format(F.col("time"), "yyyy-MM-dd"))\
 #                   .withColumn("hour", F.date_format(F.col("time"), "HH"))\
  #                  .withColumn("minute", F.date_format(F.col("time"), "mm"))\
   #                 .withColumn("second", F.date_format(F.col("time"), "ss"))
    
df_silver = df_silver.withColumn("time", F.to_timestamp(F.col("time"), "yyyy-MM-dd HH:mm:ss"))\
    .withColumn("date", F.date_format(F.col("time"), "yyyy-MM-dd"))\
    .withColumn("hour", F.date_format(F.col("time"), "HH"))\
    .withColumn("minute", F.date_format(F.col("time"), "mm"))\
    .withColumn("second", F.date_format(F.col("time"), "ss"))

# df_silver = df_silver.withColumn("time", F.to_timestamp(F.col("time"), "yyyy-MM-dd HH:mm:ss"))\
#     .withColumn("date_t", F.date_format(F.col("time"), "yyyy-MM-dd"))\
#     .withColumn("hour_t", F.date_format(F.col("time"), "HH"))\
#     .withColumn("minute_t", F.date_format(F.col("time"), "mm"))\
#     .withColumn("second_t", F.date_format(F.col("time"), "ss"))



In [0]:
df_silver=df_silver.withColumn("card_type", F.initcap(F.col("card_type")))\
    .withColumn("card_type", F.regexp_replace(F.col("card_type"), "Mastercard", "Master Card"))\
    .withColumn("amount", F.col("amount").cast("float"))\
    .withColumn("city", F.initcap(F.col("city")))

#df_silver.printSchema()
#display(df_silver)

In [0]:
#remove trailing and leading spaces
string_cols = [f.name for f in df_silver.schema.fields if f.dataType.simpleString() == "string"]

for col_name in string_cols:
    df_silver = df_silver.withColumn(col_name, F.trim(F.col(col_name)))
#Let's check the distinct values in the status column

In [0]:
#Let's find if there's anything of debt
display(df_silver)

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_transactions_3")

In [0]:
df_bronze = spark.table(f'{catalog_name}.bronze.brz_transactions_4')



In [0]:

df_silver=df_bronze.withColumn("status", F.initcap(F.col("status")))\
    .withColumn("card_type", F.regexp_replace(F.col("card_type"), "Vsa", "Visa"))\
    .withColumn("card_type", F.regexp_replace(F.col("card_type"), "MasterCard", "Master Card"))\
    .withColumn("city", F.regexp_replace(F.col("city"), "TEHRAN", "Tehran"))\
    .withColumn("city", F.regexp_replace(F.col("city"), "THR", "Tehran"))

display(df_silver)

In [0]:

df_silver = df_silver.withColumn("time", F.to_timestamp(F.col("time"), "HH:mm yyyy-MM-dd"))\
    .withColumn("date", F.date_format(F.col("time"), "yyyy-MM-dd"))\
    .withColumn("hour", F.date_format(F.col("time"), "HH"))\
    .withColumn("minute", F.date_format(F.col("time"), "mm"))

# df_silver = df_silver.withColumn("time", F.to_timestamp(F.col("time"), "HH:mm yyyy-MM-dd"))\
#     .withColumn("date_t", F.date_format(F.col("time"), "yyyy-MM-dd"))\
#     .withColumn("hour_t", F.date_format(F.col("time"), "HH"))\
#     .withColumn("minute_t", F.date_format(F.col("time"), "mm"))
display(df_silver)

In [0]:
#Create the column for debt where amount is less than 0
df_silver = df_silver.withColumn("amount", F.col("amount").cast("float"))\
    .withColumn("Debt", F.when(F.col("amount") < 0, F.col("amount")).otherwise(F.lit(None)))\
    .withColumn("Debt", F.col("Debt").cast("double"))

df_silver.printSchema()

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_transactions_4")